**Ingeniería de Características (Feature Engineering) y Creación de la ABT**

In [ ]:
import pandas as pd
import os

# 1. Cargamos los datos limpios procesados en la fase anterior
ruta_accesos = "../../datos/procesados/accesos_historico_total.csv"
ruta_socios = "../../datos/procesados/socios_historico_total.csv"

df_accesos = pd.read_csv(ruta_accesos)
df_socios = pd.read_csv(ruta_socios)

# 2. Convertimos la columna de fechas a formato datetime y calculamos la fecha mas reciente del registro
df_accesos['fecha'] = pd.to_datetime(df_accesos['fecha'])
fecha_referencia = df_accesos['fecha'].max()

# --- FASE 1: INGENIERÍA DE CARACTERÍSTICAS GEOGRÁFICAS ---

# Definimos una funcion para clasificar a los socios por zonas segun su codigo postal
def asignar_zona_proximidad(cp):
    try:
        cp_int = int(float(cp))
    except:
        return 2
    
    # Zona 0: Pineda de Mar (Residente local)
    if cp_int == 8397:
        return 0
    # Zona 1: Municipios vecinos del Maresme (Calella, Sta. Susanna, Malgrat)
    elif cp_int in [8370, 8398, 8380]:
        return 1
    # Zona 2: Otros municipios o larga distancia
    else:
        return 2

# Aplicamos la logica de proximidad y creamos una nueva columna en la tabla de socios
df_socios['zona_proximidad'] = df_socios['codigo_postal'].apply(asignar_zona_proximidad)

# --- FASE 2: FEATURE ENGINEERING RFM (Recencia y Frecuencia) ---

# Calculamos la fecha del ultimo acceso y el volumen total de visitas por socio
comportamiento = df_accesos.groupby('id_socio').agg(
    ultima_visita=('fecha', 'max'),
    total_visitas=('fecha', 'count')
).reset_index()

# Calculamos los dias exactos de ausencia restando la ultima visita a la fecha de referencia
comportamiento['dias_desde_ultima_visita'] = (fecha_referencia - comportamiento['ultima_visita']).dt.days

# Extraemos el dia de la semana (0=Lunes, 6=Domingo) para identificar los patrones de asistencia
df_accesos['dia_semana'] = df_accesos['fecha'].dt.dayofweek
dia_favorito = df_accesos.groupby('id_socio')['dia_semana'].apply(
    lambda x: x.mode()[0] if not x.mode().empty else -1
).reset_index()
dia_favorito.rename(columns={'dia_semana': 'dia_favorito'}, inplace=True)

# Unimos las metricas calculadas de comportamiento y asistencia en una unica tabla
comportamiento = pd.merge(comportamiento, dia_favorito, on='id_socio', how='left')

# --- FASE 3: UNIFICACIÓN FINAL (ABT - Analytical Base Table) ---

# Cruzamos la tabla de socios (demografia) con la de comportamiento para obtener una vision 360
df_abt = pd.merge(df_socios, comportamiento, on='id_socio', how='left')

# Imputamos valores nulos aplicando logica de negocio
df_abt['total_visitas'] = df_abt['total_visitas'].fillna(0)
df_abt['dias_desde_ultima_visita'] = df_abt['dias_desde_ultima_visita'].fillna(999) # 999 indicara a aquellos que nunca han asistido
df_abt['dia_favorito'] = df_abt['dia_favorito'].fillna(-1)

# Eliminamos variables temporales que no aportan valor al modelo predictivo
df_abt = df_abt.drop(columns=['ultima_visita'])

print("Logica completada. Tabla ABT lista en memoria.")

Lógica completada. Tabla ABT lista en memoria.


**Análisis Exploratorio Rápido de la ABT**

In [ ]:
# Comprobamos el volumen total de registros que hemos consolidado en la tabla final
print(f"Total de socios en la tabla: {len(df_abt)}\n")

# 1. Extraemos los 10 codigos postales principales para validar la distribucion geografica real
print("--- TOP 10 CÓDIGOS POSTALES ---")
print(df_abt['codigo_postal'].value_counts().head(10))

# 2. Verificamos el volumen de socios agrupados en nuestras nuevas zonas personalizadas
print("\n--- DISTRIBUCIÓN POR ZONAS ---")
print(df_abt['zona_proximidad'].value_counts().sort_index())

# 3. Calculamos la tasa de abandono (Churn) por zona para buscar primeros patrones de fuga
print("\n--- % DE CHURN (BAJAS) POR ZONA ---")
print(df_abt.groupby('zona_proximidad')['es_baja'].mean() * 100)

Total de socios en la tabla: 8658

--- TOP 10 CÓDIGOS POSTALES ---
codigo_postal
08397     3307
8397.0    2088
08370      942
8370.0     823
08395      148
08398      141
8395.0     137
8398.0     115
08396      107
8396.0      89
Name: count, dtype: int64

--- DISTRIBUCIÓN POR ZONAS ---
zona_proximidad
0    5397
1    2169
2    1092
Name: count, dtype: int64

--- % DE CHURN (BAJAS) POR ZONA ---
zona_proximidad
0    38.688160
1    46.150300
2    48.626374
Name: es_baja, dtype: float64


**Exportación Final de la ABT**

In [ ]:
# --- FASE FINAL: GUARDADO DE LA ABT ---

# 1. Definimos la ruta de destino y nos aseguramos de crear los directorios si no existen
ruta_guardado = "../../datos/procesados"
os.makedirs(ruta_guardado, exist_ok=True)

# 2. Exportamos nuestra tabla final consolidada a formato CSV, omitiendo el indice
ruta_final = f"{ruta_guardado}/abt_socios_modelo.csv"
df_abt.to_csv(ruta_final, index=False)

# 3. Imprimimos un mensaje de confirmacion con la ruta exacta del archivo generado
print(f"Archivo procesado y guardado fisicamente con exito en:\n{ruta_final}")

Archivo procesado y guardado físicamente con éxito en:
../../datos/procesados/abt_socios_modelo.csv
